<a href="https://colab.research.google.com/github/Arjx01/GenAI/blob/main/water_leak_detector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install dependencies
!pip install -q -U langchain-google-genai langchain-core pandas tabulate ipywidgets


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.0 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.0 which is incompatible.


In [2]:
import pandas as pd
import os

csv_path = 'water_consumption.csv'
if not os.path.exists(csv_path):
    csv_data = """event_id,date,device_fixture,description,liters_consumed,category,reading_type,location
EV1001,2025-01-05,Washing Machine,NORMAL COTTON CYCLE,60.00,Laundry,metered,Utility_Room
EV1002,2025-02-05,Washing Machine,NORMAL COTTON CYCLE,60.00,Laundry,metered,Utility_Room
EV1003,2025-03-05,Washing Machine,NORMAL COTTON CYCLE,60.00,Laundry,metered,Utility_Room
EV1004,2025-04-05,Washing Machine,FAULTY VALVE OVERFLOW,110.00,Laundry,metered,Utility_Room
EV1005,2025-01-10,Water Purifier RO,AUTO FLUSH CYCLE,15.00,Kitchen,metered,Kitchen_Sink
EV1006,2025-02-10,Water Purifier RO,AUTO FLUSH CYCLE,15.00,Kitchen,metered,Kitchen_Sink
EV1007,2025-03-10,Water Purifier RO,AUTO FLUSH CYCLE,15.00,Kitchen,metered,Kitchen_Sink
EV1008,2025-04-10,Water Purifier RO,AUTO FLUSH CYCLE,15.00,Kitchen,metered,Kitchen_Sink
EV1009,2024-04-15,Solar Water Heater,ANNUAL TANK FLUSH DRAIN,300.00,Utility,metered,Rooftop
EV1010,2025-04-15,Solar Water Heater,ANNUAL TANK FLUSH DRAIN,300.00,Utility,metered,Rooftop
EV1011,2025-01-02,Garden Drip System,QUARTERLY DEEP SOAK,450.00,Outdoor,metered,Backyard
EV1012,2025-04-03,Garden Drip System,QUARTERLY DEEP SOAK,450.00,Outdoor,metered,Backyard
EV1013,2025-03-18,Bathroom Shower,EVENING SHOWER,40.00,Personal,metered,Master_Bath
EV1014,2025-03-18,Bathroom Shower,EVENING SHOWER,40.00,Personal,metered,Master_Bath
EV1015,2025-03-20,Rainwater Tank Fill,MONSOON RUNOFF INFLOW,500.00,Conservation,metered,Storage_Sump
EV1016,2025-03-21,Rainwater Tank Fill,DISCARDED FIRST FLUSH DRAIN,-500.00,Conservation,drain_offset,Storage_Sump
EV1017,2025-02-14,Dishwasher,ECO WASH RUN,12.00,Kitchen,metered,Kitchen_Main
EV1018,2025-03-14,Dishwasher,ECO WASH RUN,12.00,Kitchen,metered,Kitchen_Main
EV1019,2025-04-14,Dishwasher,ECO WASH RUN,12.00,Kitchen,metered,Kitchen_Main
EV1020,2025-04-20,Swimming Pool Top-up,SEASONAL POOL REFILL,4500.00,Recreation,metered,Backyard"""
    with open(csv_path, 'w', encoding='utf-8') as f:
        f.write(csv_data)
    print(f"Created {csv_path}")
else:
    print(f"Using existing {csv_path}")

df_preview = pd.read_csv(csv_path)
display(df_preview.head())


Created water_consumption.csv


/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


,event_id,date,device_fixture,description,liters_consumed,category,reading_type,location
0,EV1001,2025-01-05,Washing Machine,NORMAL COTTON CYCLE,60.0,Laundry,metered,Utility_Room
1,EV1002,2025-02-05,Washing Machine,NORMAL COTTON CYCLE,60.0,Laundry,metered,Utility_Room
2,EV1003,2025-03-05,Washing Machine,NORMAL COTTON CYCLE,60.0,Laundry,metered,Utility_Room
3,EV1004,2025-04-05,Washing Machine,FAULTY VALVE OVERFLOW,110.0,Laundry,metered,Utility_Room
4,EV1005,2025-01-10,Water Purifier RO,AUTO FLUSH CYCLE,15.0,Kitchen,metered,Kitchen_Sink


In [3]:
import os
import re
import numpy as np
import pandas as pd
from datetime import datetime
from tabulate import tabulate

def normalize_fixture(name: str) -> str:
    """Normalize fixture/appliance names."""
    name = str(name).lower().strip()
    name = re.sub(r'\b(auto|flush|cycle|run|normal|eco|system|fill|drain)\b', '', name)
    name = re.sub(r'[^a-z0-9\s]', ' ', name)
    return " ".join(name.split()).title()

class UtilityLeakEngine:
    def __init__(self, csv_path: str):
        self.df = pd.read_csv(csv_path)
        self._clean_data()

    def _clean_data(self):
        self.df['date'] = pd.to_datetime(self.df['date'])
        self.df['liters_consumed'] = pd.to_numeric(self.df['liters_consumed'], errors='coerce').fillna(0.0)
        self.df['norm_fixture'] = self.df['device_fixture'].apply(normalize_fixture)
        self.df['reading_type'] = self.df['reading_type'].astype(str).str.lower().str.strip()
        self.df = self.df.sort_values(by='date').reset_index(drop=True)

    def detect_recurring_cycles(self):
        """Tool A: Detect automated periodic cycles (sprinklers, maintenance flushes)."""
        recurring = []
        positive_reads = self.df[self.df['liters_consumed'] > 0]

        for fixture, group in positive_reads.groupby('norm_fixture'):
            if len(group) < 2:
                continue

            dates = group['date'].tolist()
            intervals = [(dates[i] - dates[i-1]).days for i in range(1, len(dates))]
            median_gap = np.median(intervals)
            amounts = group['liters_consumed'].tolist()
            typical_amount = float(np.median(amounts))

            freq = None
            annual_multiplier = 0
            if 25 <= median_gap <= 35:
                freq = "Monthly Cycle"
                annual_multiplier = 12
            elif 80 <= median_gap <= 100:
                freq = "Quarterly Cycle"
                annual_multiplier = 4
            elif 170 <= median_gap <= 195:
                freq = "Semiannual Cycle"
                annual_multiplier = 2
            elif 350 <= median_gap <= 380:
                freq = "Annual Cycle"
                annual_multiplier = 1

            if freq:
                confidence = "High" if len(group) >= 3 else "Medium"
                last_date = dates[-1]
                next_est = last_date + pd.Timedelta(days=int(median_gap))

                recurring.append({
                    "device_fixture": fixture,
                    "cycle_frequency": freq,
                    "typical_liters": typical_amount,
                    "observed_readings": len(group),
                    "last_reading_date": last_date.strftime('%Y-%m-%d'),
                    "estimated_next_date": next_est.strftime('%Y-%m-%d'),
                    "estimated_annual_liters": typical_amount * annual_multiplier,
                    "confidence": confidence,
                    "event_ids": group['event_id'].tolist()
                })
        return recurring

    def detect_duplicate_readings(self, day_window: int = 1):
        """Tool B: Detect duplicate meter logs on the same day."""
        duplicates = []
        positive_reads = self.df[self.df['liters_consumed'] > 0]

        for fixture, group in positive_reads.groupby('norm_fixture'):
            if len(group) < 2:
                continue
            group = group.sort_values(by='date').reset_index(drop=True)
            for i in range(len(group) - 1):
                row1 = group.iloc[i]
                row2 = group.iloc[i + 1]
                date_diff = abs((row2['date'] - row1['date']).days)

                if date_diff <= day_window and abs(row1['liters_consumed'] - row2['liters_consumed']) < 0.01:
                    duplicates.append({
                        "device_fixture": fixture,
                        "liters": float(row1['liters_consumed']),
                        "event_id_1": row1['event_id'],
                        "event_id_2": row2['event_id'],
                        "date_1": row1['date'].strftime('%Y-%m-%d'),
                        "date_2": row2['date'].strftime('%Y-%m-%d'),
                        "reason": f"Same fixture and identical reading ({row1['liters_consumed']} L) within {date_diff} day(s)."
                    })
        return duplicates

    def detect_leak_spikes(self):
        """Tool C: Detect sudden consumption spikes / valve leaks."""
        spikes = []
        recurring = self.detect_recurring_cycles()
        recurring_fixtures = [r['device_fixture'] for r in recurring]

        for fixture in recurring_fixtures:
            group = self.df[self.df['norm_fixture'] == fixture].sort_values(by='date')
            amounts = group['liters_consumed'].tolist()
            dates = group['date'].tolist()
            event_ids = group['event_id'].tolist()

            for i in range(1, len(amounts)):
                if amounts[i] != amounts[i - 1]:
                    diff = amounts[i] - amounts[i - 1]
                    pct = (diff / amounts[i - 1]) * 100
                    spikes.append({
                        "device_fixture": fixture,
                        "previous_liters": float(amounts[i - 1]),
                        "latest_liters": float(amounts[i]),
                        "difference_liters": float(diff),
                        "pct_change": round(pct, 2),
                        "type": "Usage Spike / Leak" if diff > 0 else "Reduction",
                        "old_date": dates[i - 1].strftime('%Y-%m-%d'),
                        "new_date": dates[i].strftime('%Y-%m-%d'),
                        "old_event_id": event_ids[i - 1],
                        "new_event_id": event_ids[i]
                    })
        return spikes

    def recurring_summary(self):
        """Tool D: Calculate baseline household water usage."""
        subs = self.detect_recurring_cycles()
        total_annual = sum(s['estimated_annual_liters'] for s in subs)
        monthly_runrate = total_annual / 12.0

        by_fixture = {s['device_fixture']: s['estimated_annual_liters'] for s in subs}
        top_consumers = sorted(subs, key=lambda x: x['estimated_annual_liters'], reverse=True)

        return {
            "total_monthly_baseline_liters": round(monthly_runrate, 2),
            "total_annual_baseline_liters": round(total_annual, 2),
            "recurring_fixtures_count": len(subs),
            "annual_liters_by_fixture": by_fixture,
            "highest_water_consumers": [
                {"fixture": s['device_fixture'], "annual_liters": s['estimated_annual_liters'], "cycle": s['cycle_frequency']}
                for s in top_consumers[:5]
            ]
        }

    def dataset_summary(self):
        """Tool F: General meter overview."""
        total_used = self.df[self.df['liters_consumed'] > 0]['liters_consumed'].sum()
        total_offset = self.df[self.df['liters_consumed'] < 0]['liters_consumed'].sum()

        return {
            "total_meter_readings": len(self.df),
            "date_range": {
                "start": self.df['date'].min().strftime('%Y-%m-%d'),
                "end": self.df['date'].max().strftime('%Y-%m-%d')
            },
            "total_liters_metered": round(float(total_used), 2),
            "total_offset_or_drained_liters": round(abs(float(total_offset)), 2),
            "unique_fixtures_tracked": int(self.df['norm_fixture'].nunique()),
            "recurring_cycles_detected": len(self.detect_recurring_cycles()),
            "duplicate_readings_flagged": len(self.detect_duplicate_readings()),
            "consumption_spikes_flagged": len(self.detect_leak_spikes())
        }

    def investigate(self, query_term: str):
        """Tool E: Investigate a specific fixture, room, or event ID."""
        term = query_term.lower()
        matched = self.df[
            self.df['device_fixture'].str.lower().str.contains(term) |
            self.df['norm_fixture'].str.lower().str.contains(term) |
            self.df['event_id'].str.lower().str.contains(term) |
            self.df['location'].str.lower().str.contains(term) |
            self.df['description'].str.lower().str.contains(term)
        ]
        if matched.empty:
            return {"status": "Not Found", "message": f"No meter readings match '{query_term}'."}

        records = matched[['event_id', 'date', 'device_fixture', 'liters_consumed', 'category', 'location']].copy()
        records['date'] = records['date'].dt.strftime('%Y-%m-%d')
        return {"matched_count": len(records), "records": records.to_dict(orient='records')}

engine = UtilityLeakEngine(csv_path)
print("✅ Household Water Leak Engine initialized.")


✅ Household Water Leak Engine initialized.


In [4]:
print('=' * 60)
print('📊 HOUSEHOLD WATER METER SUMMARY')
print('=' * 60)
summary = engine.dataset_summary()
for k, v in summary.items():
    print(f' - {k}: {v}')

print('\n' + '=' * 60)
print('🔁 RECURRING BASELINE CYCLES')
print('=' * 60)
cycles = engine.detect_recurring_cycles()
print(tabulate(cycles, headers='keys', tablefmt='simple'))

print('\n' + '=' * 60)
print('⚠️ DUPLICATE METER LOGS')
print('=' * 60)
dups = engine.detect_duplicate_readings()
print(tabulate(dups, headers='keys', tablefmt='simple'))

print('\n' + '=' * 60)
print('📈 CONSUMPTION SPIKES & POTENTIAL LEAKS')
print('=' * 60)
spikes = engine.detect_leak_spikes()
print(tabulate(spikes, headers='keys', tablefmt='simple'))


📊 HOUSEHOLD WATER METER SUMMARY
 - total_meter_readings: 20
 - date_range: {'start': '2024-04-15', 'end': '2025-04-20'}
 - total_liters_metered: 6966.0
 - total_offset_or_drained_liters: 500.0
 - unique_fixtures_tracked: 8
 - recurring_cycles_detected: 5
 - duplicate_readings_flagged: 1
 - consumption_spikes_flagged: 1

🔁 RECURRING BASELINE CYCLES
device_fixture      cycle_frequency      typical_liters    observed_readings  last_reading_date    estimated_next_date      estimated_annual_liters  confidence    event_ids
------------------  -----------------  ----------------  -------------------  -------------------  ---------------------  -------------------------  ------------  ----------------------------------------
Dishwasher          Monthly Cycle                    12                    3  2025-04-14           2025-05-13                                   144  High          ['EV1017', 'EV1018', 'EV1019']
Garden Drip         Quarterly Cycle                 450                    2  2

In [5]:
import json
import os
import time
import getpass
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage
from langchain_google_genai import ChatGoogleGenerativeAI

# 1. Google API Key
if not os.environ.get("GOOGLE_API_KEY"):
    api_key = getpass.getpass("Enter your GOOGLE_API_KEY (press Enter to skip): ").strip()
    if api_key:
        os.environ["GOOGLE_API_KEY"] = api_key

# 2. Tools
@tool
def get_dataset_summary() -> dict:
    """Returns total readings count, total liters, date range, and candidate counts."""
    return engine.dataset_summary()

@tool
def get_recurring_cycles() -> list:
    """Returns baseline recurring cycles with frequency, typical liters, and estimated annual usage."""
    return engine.detect_recurring_cycles()

@tool
def get_duplicate_readings() -> list:
    """Returns duplicate meter readings logged on the same day."""
    return engine.detect_duplicate_readings()

@tool
def get_leak_spikes() -> list:
    """Returns fixtures that experienced sudden consumption spikes or potential leaks."""
    return engine.detect_leak_spikes()

@tool
def get_baseline_spending_analysis() -> dict:
    """Returns monthly run-rate, annual liters, and highest water-consuming fixtures."""
    return engine.recurring_summary()

@tool
def investigate_entity(query_term: str) -> dict:
    """Retrieves actual records for a specific fixture name, event ID, room, or keyword."""
    return engine.investigate(query_term)

tools = [
    get_dataset_summary,
    get_recurring_cycles,
    get_duplicate_readings,
    get_leak_spikes,
    get_baseline_spending_analysis,
    investigate_entity
]

def extract_message_text(content) -> str:
    if isinstance(content, str):
        return content
    elif isinstance(content, list):
        parts = []
        for block in content:
            if isinstance(block, str):
                parts.append(block)
            elif isinstance(block, dict):
                parts.append(block.get("text", str(block)))
            elif hasattr(block, "text"):
                parts.append(getattr(block, "text"))
            else:
                parts.append(str(block))
        return "\n".join(parts)
    return str(content)

def get_agent_for_model(model_name: str):
    llm = ChatGoogleGenerativeAI(
        model=model_name,
        max_retries=2
    )
    return llm.bind_tools(tools)

FALLBACK_MODELS = [
    os.environ.get("GEMINI_MODEL", "gemini-3.5-flash-lite"),
    "gemini-2.5-flash-lite",
    "gemini-3.8-flash",
    "gemini-2.5-flash"
]

SYSTEM_PROMPT = """You are the Household Smart Water & Utility Leak Detector AI Agent.
Rules:
1. Always base answers strictly on tool outputs. Never invent or assume readings, fixtures, or liters.
2. Distinguish clearly between observed facts, algorithmic estimates, and user confirmations.
3. Cite event IDs, dates, and liters consumed when explaining findings.
4. If asked about fixing a leak or closing valves, advise the user; never claim to have shut off any valve.
5. If the chosen model or API is unavailable, report the issue clearly.
"""

def ask_agent(user_query: str):
    if not os.environ.get("GOOGLE_API_KEY"):
        print("❌ Error: GOOGLE_API_KEY is not set.")
        return

    tool_map = {t.name: t for t in tools}

    for model_name in FALLBACK_MODELS:
        try:
            active_agent = get_agent_for_model(model_name)
            messages = [
                SystemMessage(content=SYSTEM_PROMPT),
                HumanMessage(content=user_query)
            ]
            response = active_agent.invoke(messages)

            if response.tool_calls:
                messages.append(response)
                for tool_call in response.tool_calls:
                    selected_tool = tool_map[tool_call["name"]]
                    tool_output = selected_tool.invoke(tool_call["args"])
                    print(f"🔧 [Tool Call] {tool_call['name']}({tool_call['args']})")
                    messages.append(ToolMessage(
                        tool_call_id=tool_call["id"],
                        content=json.dumps(tool_output)
                    ))

                final_res = active_agent.invoke(messages)
                text_output = extract_message_text(final_res.content)
                print(f"\n🤖 AI Agent Response ({model_name}):\n" + text_output)
            else:
                text_output = extract_message_text(response.content)
                print(f"\n🤖 AI Agent Response ({model_name}):\n" + text_output)
            return

        except Exception as e:
            err_str = str(e)
            if any(term in err_str for term in ["503", "UNAVAILABLE", "429", "RESOURCE_EXHAUSTED"]):
                print(f"⚠️ {model_name} is temporarily busy. Trying next model...")
                time.sleep(1)
                continue
            else:
                print(f"❌ Error with {model_name}: {err_str}")
                return

print("✅ Smart Water Leak Agent and tools configured.")


Enter your GOOGLE_API_KEY (press Enter to skip): ··········
✅ Smart Water Leak Agent and tools configured.


In [6]:
ask_agent('Which household appliances consume the most water, and did any fixture show a sudden spike or leak?')

🔧 [Tool Call] get_baseline_spending_analysis({})
🔧 [Tool Call] get_leak_spikes({})

🤖 AI Agent Response (gemini-3.5-flash-lite):
Based on the utility data analysis, here are the details regarding household water consumption and potential leaks:

### Highest Water-Consuming Household Fixtures & Appliances
According to the baseline spending analysis, the household fixtures and appliances ranked by their annual baseline consumption are:
1. **Garden Drip**: 1,800 liters annually (Quarterly Cycle)
2. **Washing Machine**: 720 liters annually (Monthly Cycle)
3. **Solar Water Heater**: 300 liters annually (Annual Cycle)
4. **Water Purifier RO**: 180 liters annually (Monthly Cycle)
5. **Dishwasher**: 144 liters annually (Monthly Cycle)

---

### Sudden Spikes or Leaks Detected
Yes, an algorithmic anomaly/spike was detected:
* **Fixture:** Washing Machine
* **Event ID:** EV1004 (compared against previous event EV1003)
* **Date:** 2025-04-05 (previous reading on 2025-03-05)
* **Consumption Change